# Prediksi Luaran Pasien (Sembuh vs Meninggal) Berbasis Rekam Medis & Komorbiditas
**MBKM Machine Learning · Ivan Nur Azmi (123430119) · Teknik Biomedis Itera**

---

### Deskripsi & Konteks Ilmiah
Penggunaan **PyCaret** membuat alur kerja *Machine Learning* menjadi jauh lebih ringkas dan rapi dibandingkan menulis *pipeline* scikit-learn dan imblearn secara manual dari nol. Proses seperti *One-Hot Encoding*, *Standard Scaling*, pembuatan *Stratified Group Cross-Validation*, komparasi banyak model sekaligus, penyetelan *threshold* probabilitas, hingga visualisasi kurva evaluasi (ROC, PR, *Confusion Matrix*, *Feature Importance*) dan penyimpanan model dapat diringkas hanya dalam beberapa baris fungsi utama (`setup`, `compare_models`, `plot_model`, `predict_model`, `save_model`).

Agar metodologi ilmiah dan validitas klinis dari penelitian di notebook sebelumnya tetap terjaga sepenuhnya, notebook ini mengintegrasikan tahapan pembersihan data & ekstraksi fitur patofisiologis yang sama persis, lalu menyerahkan seluruh beban *preprocessing*, pemodelan *Stratified Group K-Fold*, dan evaluasi ke ekosistem PyCaret (bebas *target leakage* dan *pseudoreplication*).

In [1]:
pip install pycaret openpyxl scipy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Bagian 1 — Import Libraries & Konfigurasi

In [2]:
import os
import re
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold

# Import modul klasifikasi PyCaret
from pycaret.classification import (
    setup, compare_models, create_model, tune_model,
    plot_model, evaluate_model, predict_model,
    finalize_model, save_model, load_model, pull
)

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Menyiapkan folder output untuk menyimpan artefak evaluasi dan model
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("Semua library dan modul PyCaret berhasil dimuat")

Semua library dan modul PyCaret berhasil dimuat


---
## Bagian 2 — Memuat Dataset & Kriteria Inklusi Ketat

Menerapkan kriteria inklusi ilmiah yang konsisten dengan penelitian rekam medis retrospektif:
1. Luaran pasien wajib tercatat secara valid (`Sembuh` atau `Meninggal`).
2. Diagnosis klinis wajib terisi (tidak `NaN`).
3. Eksklusi sampel non-kultur atau artefak lab (`CANCELLED` atau `PARAINFLUENZA`).

In [3]:
file_path = Path('dataset/raw/dataset_gabungan_luaran.xlsx').resolve()
df_raw = pd.read_excel(file_path, engine='openpyxl')

print(f"Dimensi dataset mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")

# Kriteria Inklusi Ketat:
df = df_raw.copy()
df['Luaran_clean'] = df['Luaran'].astype(str).str.strip().str.capitalize()

kondisi_valid = (
    df['Luaran_clean'].isin(['Sembuh', 'Meninggal']) &
    df['Diagnosis'].notna() &
    (~df['Nama bakteri'].astype(str).str.contains('CANCELLED|PARAINFLUENZA', case=False, na=False))
)

df = df[kondisi_valid].copy().reset_index(drop=True)
print(f"Baris memenuhi syarat inklusi: {len(df)} baris")
print("\nDistribusi Target Luaran:")
print(df['Luaran_clean'].value_counts())

Dimensi dataset mentah: 399 baris x 9 kolom
Baris memenuhi syarat inklusi: 223 baris

Distribusi Target Luaran:
Luaran_clean
Sembuh       185
Meninggal     38
Name: count, dtype: int64


---
## Bagian 3 — Feature Engineering Klinis & Rekonstruksi ID Pasien Unik

Tahap ini tetap dilakukan dengan Regular Expressions (*Regex*) untuk mengekstrak sindrom klinis dari teks bebas dan mencegah *target leakage* maupun *administrative confounding*:
1. **Pembersihan Klausa Post-Mortem ICD:** Menghapus frasa `with death due to brain injury` agar tidak terjadi kebocoran informasi kematian retrospektif (*future leakage*).
2. **Ekstraksi 7 Sindrom Klinis Murni:** Sepsis, Gagal Ginjal, Diabetes Melitus, Stroke/ICH, Gagal Napas, Penyakit Kardiovaskular, dan Penyakit Hati.
3. **Kategorisasi Patogen & Spesimen:** Mengelompokkan jenis sampel dan nama bakteri ke dalam kelas klinis dominan.
4. **Rekonstruksi True Patient ID:** Mengatasi kelemahan pencatatan ID 7-digit berulang guna mencegah kebocoran data antar-isolat dari pasien yang sama (*pseudoreplication*).

In [4]:
# 1. Pembersihan eksplisit klausa post-mortem ICD ("with death due to brain injury")
df['Diagnosis_clean'] = df['Diagnosis'].fillna('').astype(str).str.replace(
    r'with death due to brain injury[,\w\s]*', '', regex=True, case=False
)
df['Komorbid_clean'] = df['Komorbid'].fillna('').astype(str)
df['combined_text'] = (df['Diagnosis_clean'] + ' ' + df['Komorbid_clean']).str.lower()

# 2. Ekstraksi 7 Sindrom Klinis Murni (bebas bias target leakage)
df['is_sepsis'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(sepsis|septic|septik|septicemia|septikemia|syok\s+septik|septic\s+shock)\b', x) else 0
)
df['is_kidney_failure'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(ckd|aki|renal|azotemia|ginjal|kidney|hepatorenal|nephro\w*|nefr\w*)\b', x) else 0
)
df['is_diabetes'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(niddm|iddm|diabetes|dm|hiperglikemia|hyperglycemia)\b', x) else 0
)
df['is_stroke_ich'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(ich|stroke|cerebri|cerebral|snh|intracranial|cva|subarachnoid)\b', x) else 0
)
df['is_respiratory_failure'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(respiratory failure|gagal napas|gagal nafas|ards|edema paru|pulmonary edema|pneumonia)\b', x) else 0
)
df['is_cardiovascular'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(heart failure|gagal jantung|cardiogenic|infark miokard|myocardial infarction|cor pulmonale)\b', x) else 0
)
df['is_liver_disease'] = df['combined_text'].apply(
    lambda x: 1 if re.search(r'\b(hepatic|liver|sirosis|cirrhosis|hepatorenal|hepatitis|hati)\b', x) else 0
)

# 3. Kategorisasi Spesimen & Patogen
top_spesimen = ['urin', 'sputum', 'darah', 'jaringan', 'abses']
df['spesimen_kategori'] = df['Jenis sampel'].apply(
    lambda val: str(val).strip().lower() if pd.notna(val) and str(val).strip().lower() in top_spesimen else 'lainnya'
)

def clean_patogen_v2(val):
    v = str(val).lower() if pd.notna(val) else 'lainnya'
    if 'klebsiella' in v: return 'klebsiella'
    elif 'coli' in v or 'escherichia' in v: return 'e_coli'
    elif 'acinetobacter' in v: return 'acinetobacter'
    elif 'staph' in v and ('aureus' in v or 'coag +' in v): return 'staph_aureus'
    elif 'staph' in v: return 'staph_cons'
    elif 'yeast' in v or 'candida' in v: return 'candida_yeast'
    elif 'pseudomonas' in v: return 'pseudomonas'
    else: return 'lainnya'

df['patogen_kategori'] = df['Nama bakteri'].apply(clean_patogen_v2)
df['target_luaran'] = df['Luaran_clean'].map({'Meninggal': 1, 'Sembuh': 0})

# 4. Rekonstruksi True Patient ID (Mencegah bocornya isolat berulang dari pasien yang sama)
df['patient_id_raw'] = df['ID sampel'].astype(str).str.strip()
is_7digit = df['patient_id_raw'].str.match(r'^\d{6,8}$')
adm_signature = (
    df['Diagnosis'].fillna('').str.strip() + '|' +
    df['Komorbid'].fillna('').str.strip() + '|' +
    df['Lama rawat inap'].fillna('').astype(str).str.strip()
)
df['true_patient_id'] = np.where(
    is_7digit,
    np.char.add('ADM_', adm_signature.factorize()[0].astype(str)),
    df['patient_id_raw']
)

print(f"Total baris isolat: {len(df)} | Total pasien unik: {df['true_patient_id'].nunique()}")
print(f"Distribusi Target Pasien Unik: {df.groupby('true_patient_id')['target_luaran'].max().value_counts().to_dict()}")

Total baris isolat: 223 | Total pasien unik: 83
Distribusi Target Pasien Unik: {0: 67, 1: 16}


---
## Bagian 4 — Persiapan Data & Stratified Group K-Fold Splitting

Untuk menjamin validitas evaluasi *hold-out test set* tanpa kebocoran data (*data leakage*), pembagian data latih (80%) dan data uji (20%) dilakukan berbasis `StratifiedGroupKFold` dengan kelompok pasien `true_patient_id`.

Dengan skema ini, seluruh riwayat isolat dari seorang pasien tertentu dipastikan masuk secara utuh ke data latih saja atau data uji saja.

In [5]:
fitur_prediktor = [
    'is_respiratory_failure', 'is_kidney_failure', 'is_sepsis',
    'is_diabetes', 'is_stroke_ich', 'is_cardiovascular', 'is_liver_disease',
    'spesimen_kategori', 'patogen_kategori'
]

# Menyiapkan dataframe khusus untuk dimasukkan ke PyCaret
df_pycaret = df[fitur_prediktor + ['target_luaran', 'true_patient_id']].copy()

# Pembagian Train (80%) dan Test Hold-out (20%) berbasis StratifiedGroupKFold agar pasien tidak bocor
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(sgkf.split(df_pycaret, df_pycaret['target_luaran'], df_pycaret['true_patient_id']))

df_train = df_pycaret.iloc[train_idx].reset_index(drop=True)
df_test = df_pycaret.iloc[test_idx].reset_index(drop=True)

print(f"Data Latih : {len(df_train)} isolat ({df_train['true_patient_id'].nunique()} pasien unik)")
print(f"Data Uji   : {len(df_test)} isolat ({df_test['true_patient_id'].nunique()} pasien unik)")
print(f"Distribusi Target Data Latih : {df_train['target_luaran'].value_counts().to_dict()}")
print(f"Distribusi Target Data Uji   : {df_test['target_luaran'].value_counts().to_dict()}")

Data Latih : 180 isolat (68 pasien unik)
Data Uji   : 43 isolat (15 pasien unik)
Distribusi Target Data Latih : {0: 148, 1: 32}
Distribusi Target Data Uji   : {0: 37, 1: 6}


---
## Bagian 5 — Inisialisasi Pipeline Otomatis dengan `setup()` PyCaret

Cukup dengan satu fungsi `setup()`, PyCaret otomatis menangani:
- **One-Hot Encoding** untuk fitur kategorikal (`spesimen_kategori`, `patogen_kategori`).
- **Normalisasi/Penskalaan Standar** (`normalize=True, normalize_method='zscore'`).
- **Penanganan Class Imbalance** (`fix_imbalance=True`) menggunakan SMOTE di dalam setiap fold CV pelatihan secara mandiri.
- **Stratified Group Cross-Validation**: Menggunakan `StratifiedGroupKFold` dengan parameter `fold_groups='true_patient_id'` sehingga ID pasien digunakan murni sebagai unit klaster validasi dan otomatis dikecualikan dari fitur pelatihan model.

In [6]:
exp_clf = setup(
    data=df_train,
    test_data=df_test,
    target='target_luaran',
    index=False,
    ignore_features=['true_patient_id'],
    fold_groups='true_patient_id',
    categorical_features=['spesimen_kategori', 'patogen_kategori'],
    numeric_features=[
        'is_respiratory_failure', 'is_kidney_failure', 'is_sepsis',
        'is_diabetes', 'is_stroke_ich', 'is_cardiovascular', 'is_liver_disease'
    ],
    normalize=True,
    normalize_method='zscore',
    fix_imbalance=True,  # Otomatis menyeimbangkan kelas minoritas (Meninggal) di tiap fold CV latih
    fold_strategy=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    fold=5,
    session_id=RANDOM_SEED,
    verbose=True
)

  File "C:\Users\user\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Python\pythoncore-3.11-64\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Python\pythoncore-3.11-64\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\user\AppData\Local\Python\pythoncore-3.11-64\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


,Description,Value
0,Session id,42
1,Target,target_luaran
2,Target type,Binary
3,Original data shape,"(223, 11)"
4,Transformed data shape,"(339, 22)"
5,Transformed train set shape,"(296, 22)"
6,Transformed test set shape,"(43, 22)"
7,Ignore features,1
8,Numeric features,7
9,Categorical features,2


---
## Bagian 6 — Benchmark Algoritma Sekaligus dengan `compare_models()`

Membandingkan 6 algoritma klasifikasi yang sepadan dengan penelitian sebelumnya:
- **Random Forest (`'rf'`)**
- **Gradient Boosting (`'gbc'`)**
- **Logistic Regression (`'lr'`)**
- **Decision Tree (`'dt'`)**
- **Support Vector Machine RBF (`'rbfsvm'`)** *(SVC RBF dengan estimasi probabilitas aktif)*
- **K-Nearest Neighbors (`'knn'`)**

Hasil dievaluasi pada 5-Fold Stratified Group CV dan diurutkan berdasarkan metrik **ROC-AUC** tertinggi.

In [7]:
# Membandingkan algoritma yang setara dengan notebook sebelumnya, diurutkan berdasarkan ROC-AUC tertinggi
# Catatan: 'rbfsvm' digunakan untuk merepresentasikan SVC(kernel='rbf', probability=True)
best_model = compare_models(
    include=['rf', 'gbc', 'lr', 'dt', 'rbfsvm', 'knn'],
    sort='AUC'
)

# Mengambil dan menampilkan tabel hasil benchmark CV
df_hasil_cv = pull()
print("\nRingkasan Perbandingan Model (5-Fold Stratified Group CV):")
display(df_hasil_cv)

# Menyimpan tabel benchmark ke CSV
df_hasil_cv.to_csv('output/benchmark_pycaret_cv.csv', index=False)
print(f"\nModel Terbaik Terpilih: {best_model}")

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8200,0.6872,0.2709,0.4857,0.3103,0.2211,0.2496,1.7420
lr,Logistic Regression,0.7833,0.6768,0.2055,0.3511,0.1981,0.1096,0.1429,0.1120
rbfsvm,SVM - Radial Kernel,0.7856,0.6657,0.1390,0.3150,0.1722,0.0758,0.1008,0.0700
gbc,Gradient Boosting Classifier,0.7728,0.5701,0.1747,0.3879,0.2143,0.1123,0.1398,0.2140
knn,K Neighbors Classifier,0.7095,0.5669,0.4544,0.3156,0.2800,0.1633,0.2121,0.0620
dt,Decision Tree Classifier,0.8216,0.4609,0.2016,0.5000,0.2403,0.1560,0.2042,0.1060



Ringkasan Perbandingan Model (5-Fold Stratified Group CV):


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8200,0.6872,0.2709,0.4857,0.3103,0.2211,0.2496,1.742
lr,Logistic Regression,0.7833,0.6768,0.2055,0.3511,0.1981,0.1096,0.1429,0.112
rbfsvm,SVM - Radial Kernel,0.7856,0.6657,0.1390,0.3150,0.1722,0.0758,0.1008,0.070
gbc,Gradient Boosting Classifier,0.7728,0.5701,0.1747,0.3879,0.2143,0.1123,0.1398,0.214
knn,K Neighbors Classifier,0.7095,0.5669,0.4544,0.3156,0.2800,0.1633,0.2121,0.062
dt,Decision Tree Classifier,0.8216,0.4609,0.2016,0.5000,0.2403,0.1560,0.2042,0.106



Model Terbaik Terpilih: RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='sqrt',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_samples_leaf=1,
                       min_samples_split=2, min_weight_fraction_leaf=0.0,
                       monotonic_cst=None, n_estimators=100, n_jobs=-1,
                       oob_score=False, random_state=42, verbose=0,
                       warm_start=False)


---
## Bagian 7 — Hyperparameter Tuning & Evaluasi Visual Otomatis

Kita melakukan *hyperparameter tuning* pada model terbaik terpilih untuk memaksimalkan metrik **AUC**.
Skema CV dan grup pasien tetap diwarisi dari konfigurasi `setup()`. Parameter `choose_better=True` memastikan bahwa jika hasil tuning tidak meningkatkan performa, PyCaret akan mempertahankan model dasar yang lebih unggul.

Selanjutnya disajikan 5 grafik diagnostik klinis utama (`auc`, `pr`, `confusion_matrix`, `feature`, dan `calibration`).

In [8]:
# Hyperparameter tuning pada model terbaik untuk mengoptimalkan AUC
tuned_best_model = tune_model(
    best_model,
    optimize='AUC',
    choose_better=True
)

# Menampilkan metrik performa setelah tuning
df_tuned_metrics = pull()
print("\nPerforma Model Setelah Hyperparameter Tuning:")
display(df_tuned_metrics)

# 1. Plot Kurva ROC-AUC
try:
    plot_model(tuned_best_model, plot='auc', save=True)
except Exception as e:
    print(f"Plot ROC-AUC dilewati: {e}")

# 2. Plot Kurva Precision-Recall (PR-AUC)
try:
    plot_model(tuned_best_model, plot='pr', save=True)
except Exception as e:
    print(f"Plot PR-AUC dilewati: {e}")

# 3. Plot Confusion Matrix
try:
    plot_model(tuned_best_model, plot='confusion_matrix', save=True)
except Exception as e:
    print(f"Plot Confusion Matrix dilewati: {e}")

# 4. Plot Feature Importance (Faktor Risiko Klinis Paling Berpengaruh)
try:
    plot_model(tuned_best_model, plot='feature', save=True)
except Exception as e:
    print(f"Plot Feature Importance tidak tersedia untuk tipe model ini: {e}")

# 5. Plot Kurva Kalibrasi Probabilitas
try:
    plot_model(tuned_best_model, plot='calibration', save=True)
except Exception as e:
    print(f"Plot Kalibrasi dilewati: {e}")

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9762,0.9934,0.7500,1.0000,0.8571,0.8444,0.8549
1,0.9250,0.9487,0.0000,0.0000,0.0000,-0.0345,-0.0367
2,0.6111,0.7492,0.2308,0.4286,0.3000,0.0632,0.0690
3,0.9474,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.7209,0.8424,0.1429,1.0000,0.2500,0.1835,0.3179
Mean,0.8361,0.7067,0.2247,0.4857,0.2814,0.2113,0.2410
Std,0.1441,0.3634,0.2770,0.4481,0.3134,0.3252,0.3309


Fitting 5 folds for each of 10 candidates, totalling 50 fits

Performa Model Setelah Hyperparameter Tuning:


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9762,0.9934,0.7500,1.0000,0.8571,0.8444,0.8549
1,0.9250,0.9487,0.0000,0.0000,0.0000,-0.0345,-0.0367
2,0.6111,0.7492,0.2308,0.4286,0.3000,0.0632,0.0690
3,0.9474,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.7209,0.8424,0.1429,1.0000,0.2500,0.1835,0.3179
Mean,0.8361,0.7067,0.2247,0.4857,0.2814,0.2113,0.2410
Std,0.1441,0.3634,0.2770,0.4481,0.3134,0.3252,0.3309


---
## Bagian 8 — Optimasi Ambang Batas Klinis (*Threshold Optimization*) & Prediksi Data Uji

Dalam kasus mortalitas infeksi di rumah sakit, ambang batas *default* `0.50` sering kali kurang sensitif mendeteksi pasien berisiko tinggi.

Pada PyCaret 3.x, kita mengonfigurasi `probability_threshold=0.35` secara langsung di fungsi `predict_model()` untuk meningkatkan Sensitivitas/Recall, sehingga dokter dapat melakukan intervensi dini secara proaktif.
Evaluasi dilakukan pada isolat data uji dan diagregasikan ke level **pasien unik**.

In [9]:
# Prediksi pada Data Uji (Hold-out Test Set) menggunakan probability_threshold klinis (0.35)
CLINICAL_THRESHOLD = 0.35

pred_test = predict_model(
    tuned_best_model,
    data=df_test,
    probability_threshold=CLINICAL_THRESHOLD
)

# Normalisasi nama kolom prediksi agar kompatibel dengan PyCaret 2.x & 3.x
if 'Label' in pred_test.columns and 'prediction_label' not in pred_test.columns:
    pred_test['prediction_label'] = pred_test['Label']
if 'Score' in pred_test.columns and 'prediction_score' not in pred_test.columns:
    pred_test['prediction_score'] = pred_test['Score']

# Menampilkan metrik evaluasi pada data uji
df_metrik_uji = pull()
print(f"\nPerforma Model pada Data Uji (Threshold = {CLINICAL_THRESHOLD}):")
display(df_metrik_uji)

# Menampilkan 10 baris hasil prediksi isolat data uji
print("\nContoh 10 Baris Hasil Prediksi pada Data Uji:")
display(pred_test[['true_patient_id', 'target_luaran', 'prediction_label', 'prediction_score']].head(10))

# Evaluasi Agregasi di Level Pasien Unik pada Data Uji
pat_test_eval = pred_test.groupby('true_patient_id').agg({
    'target_luaran': 'max',
    'prediction_label': 'max',
    'prediction_score': 'mean'
}).reset_index()

print(f"\nEvaluasi Matriks Kontingensi Level Pasien Unik pada Data Uji (n = {len(pat_test_eval)} pasien):")
display(pd.crosstab(
    pat_test_eval['target_luaran'],
    pat_test_eval['prediction_label'],
    rownames=['Aktual Pasien (0=Sembuh, 1=Meninggal)'],
    colnames=['Prediksi Pasien (0=Sembuh, 1=Meninggal)']
))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.8140,0.8604,0.5000,0.3750,0.4286,0.3202,0.3249



Performa Model pada Data Uji (Threshold = 0.35):


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.814,0.8604,0.5,0.375,0.4286,0.3202,0.3249



Contoh 10 Baris Hasil Prediksi pada Data Uji:


,true_patient_id,target_luaran,prediction_label,prediction_score
0,050D,1,1,0.5207
1,002D,1,1,0.5442
2,ADM_15,0,0,0.6654
3,ADM_17,1,0,0.7169
4,ADM_23,0,0,0.7759
5,ADM_29,0,0,0.9439
6,ADM_34,0,0,0.9671
7,ADM_35,0,0,0.9671
8,ADM_35,0,0,0.9671
9,ADM_35,0,0,0.9703



Evaluasi Matriks Kontingensi Level Pasien Unik pada Data Uji (n = 15 pasien):


"Prediksi Pasien (0=Sembuh, 1=Meninggal)",0,1
"Aktual Pasien (0=Sembuh, 1=Meninggal)",,
0,10,2
1,0,3


---
## Bagian 9 — Finalisasi & Ekspor Pipeline Produksi Mandiri (`save_model`)

Sebelum implementasi produksi, fungsi `finalize_model()` melatih ulang model terbaik pada keseluruhan dataset (Data Latih + Data Uji) untuk memanfaatkan seluruh bukti empiris yang ada.

Fungsi `save_model()` membungkus seluruh tahapan *preprocessing* (One-Hot Encoding, StandardScaler, imputasi) dan model klasifikasi ke dalam satu file berkas `.pkl` siap pakai.

In [ ]:
# Melatih ulang model terbaik pada keseluruhan dataset (Train + Test) sebelum diekspor ke produksi
all_groups = pd.concat([df_train['true_patient_id'], df_test['true_patient_id']]).reset_index(drop=True)
final_production_pipeline = finalize_model(tuned_best_model, groups=all_groups)

# Menyimpan seluruh pipeline + model menjadi file .pkl
model_export_path = 'models/pipeline_stratifikasi_risiko_pycaret'
save_model(final_production_pipeline, model_export_path)
print(f"\nPipeline produksi berhasil disimpan ke: '{model_export_path}.pkl'!")

---
## Bagian 10 — Simulasi Inferensi Pasien Baru (Clinical Decision Support System / CDSS)

Saat diimplementasikan pada sistem rekam medis elektronik rumah sakit, petugas cukup memuat berkas pipeline `.pkl` dengan `load_model()` dan memasukkan data klinis pasien baru tanpa perlu rekayasa fitur manual.

In [10]:
# Memuat kembali pipeline produksi yang telah disimpan
loaded_pipeline = load_model('models/pipeline_stratifikasi_risiko_pycaret')

# Contoh 2 profil pasien baru saat admisi:
# Pasien 1: Pasien kritis komorbid berat (Gagal Napas + Gagal Ginjal + Sepsis + DM) dengan kultur darah Klebsiella
# Pasien 2: Pasien risiko rendah tanpa komorbid kritis dengan kultur urin E. coli
pasien_baru = pd.DataFrame([
    {
        'is_respiratory_failure': 1,
        'is_kidney_failure': 1,
        'is_sepsis': 1,
        'is_diabetes': 1,
        'is_stroke_ich': 0,
        'is_cardiovascular': 0,
        'is_liver_disease': 0,
        'spesimen_kategori': 'darah',
        'patogen_kategori': 'klebsiella'
    },
    {
        'is_respiratory_failure': 0,
        'is_kidney_failure': 0,
        'is_sepsis': 0,
        'is_diabetes': 0,
        'is_stroke_ich': 0,
        'is_cardiovascular': 0,
        'is_liver_disease': 0,
        'spesimen_kategori': 'urin',
        'patogen_kategori': 'e_coli'
    }
])

hasil_inferensi = predict_model(
    loaded_pipeline,
    data=pasien_baru,
    probability_threshold=CLINICAL_THRESHOLD
)

# Normalisasi nama kolom prediksi
lbl_col = 'prediction_label' if 'prediction_label' in hasil_inferensi.columns else 'Label'
score_col = 'prediction_score' if 'prediction_score' in hasil_inferensi.columns else 'Score'

print("\nHasil Prediksi Simulasi Pasien Baru (CDSS):")
for idx, row in hasil_inferensi.iterrows():
    pred_val = row[lbl_col]
    score_val = row[score_col]
    status_pred = "RISIKO TINGGI (MENINGGAL)" if pred_val == 1 else "RISIKO RENDAH (SEMBUH)"
    print(f"Pasien #{idx+1} | Spesimen: {row['spesimen_kategori'].upper():<7} | Patogen: {row['patogen_kategori']:<12} -> Prediksi: {status_pred:<24} (Skor: {score_val:.3f})")

Transformation Pipeline and Model Successfully Loaded



Hasil Prediksi Simulasi Pasien Baru (CDSS):
Pasien #1 | Spesimen: DARAH   | Patogen: klebsiella   -> Prediksi: RISIKO TINGGI (MENINGGAL) (Skor: 0.940)
Pasien #2 | Spesimen: URIN    | Patogen: e_coli       -> Prediksi: RISIKO RENDAH (SEMBUH)   (Skor: 1.000)
